# 06 — Financial Information Extraction

**FinRAG learning series · Notebook 6 of 14**

## Learning objectives
1. Understand FinRAG's **canonical metric definitions** and how row labels are matched to them.
2. See why *similar-looking* items (revenue from operations vs total income; profit after tax vs profit for the year) must stay distinct.
3. Use **row-group context** to resolve ambiguous labels (current vs non-current *Borrowings*).
4. Produce structured JSON output where every value carries its printed form, unit, period, basis and source location.
5. Check extraction against an independent, hand-verified gold list.

## Prerequisites
Notebooks 01–02. A PDF with financial statements (saved outputs: pages 99–100 of a public annual report).

## Concept: deterministic extraction
FinRAG extracts financial values **without an LLM**. For each table row:
1. normalise the label (`"a) Owners of the Company"` → `"owners of the company"`);
2. match it against ~48 canonical definitions (ordered regular expressions, specific before general);
3. apply row-group rules when a label is ambiguous;
4. emit one value per numeric cell, with the cell's column header as the period and the nearest unit statement as the unit.

Why not let an LLM do it? Determinism, auditability and speed: the same input always gives the same output, and every rule can be
read and tested. (Notebook 13 shows what a small fine-tuned model does on the same task — and why it is not used in production.)

In [1]:
# Setup: make the production backend importable and keep secrets out of outputs
import sys, os
sys.path.insert(0, os.path.abspath("."))      # run notebooks from the notebooks/ folder
import finrag_nb as nb                          # helper: paths, safe settings, DB session
print("Backend package:", nb.BACKEND_DIR)
print("Documents folder:", nb.DOCS_DIR)

Backend package: /home/user/Ready_To_Deploy/backend
Documents folder: /opt/finrag_testdocs


## 1. Label normalisation and matching

In [2]:
from app.financial_analytics.metric_extractor import normalise_label, match_metric

for label in ["Revenue from Operations", "Total Income", "Profit After Tax", "Profit for the year",
              "a) Owners of the Company", "Basic (in C) 31", "Cash and Cash Equivalents", "Value of Sales & Services (Revenue)"]:
    m = match_metric(label, [])
    print(f"{label!r:<40} -> normalised {normalise_label(label)!r:<36} -> {m.key if m else None}")

'Revenue from Operations'                -> normalised 'revenue from operations'            -> revenue_from_operations
'Total Income'                           -> normalised 'total income'                       -> total_income
'Profit After Tax'                       -> normalised 'profit after tax'                   -> profit_after_tax
'Profit for the year'                    -> normalised 'profit for the year'                -> net_profit
'a) Owners of the Company'               -> normalised 'owners of the company'              -> None
'Basic (in C) 31'                        -> normalised 'basic'                              -> eps_basic
'Cash and Cash Equivalents'              -> normalised 'cash and cash equivalents'          -> cash_and_cash_equivalents
'Value of Sales & Services (Revenue)'    -> normalised 'value of sales and services (revenue)' -> None


*Value of Sales & Services (Revenue)* deliberately matches **nothing**: it is a gross figure before GST, not revenue from operations.
Similar names are never merged silently.

## 2. Ambiguous labels need their row group

In [3]:
cases = [("Borrowings", ["Liabilities", "Non-Current Liabilities", "Financial Liabilities"]),
         ("Borrowings", ["Current Liabilities", "Financial Liabilities"]),
         ("b) Non-Controlling Interest", ["Net Profit Attributable to:"]),
         ("Non-Controlling Interest", ["Equity"]),
         ("a) Owners of the Company", ["Other Comprehensive Income Attributable to:"])]
for label, groups in cases:
    m = match_metric(label, groups)
    print(f"{label:<30} groups={groups[-2:]} -> {m.key if m else None}")

Borrowings                     groups=['Non-Current Liabilities', 'Financial Liabilities'] -> borrowings_non_current
Borrowings                     groups=['Current Liabilities', 'Financial Liabilities'] -> borrowings_current
b) Non-Controlling Interest    groups=['Net Profit Attributable to:'] -> net_profit_attributable_nci
Non-Controlling Interest       groups=['Equity'] -> non_controlling_interest
a) Owners of the Company       groups=['Other Comprehensive Income Attributable to:'] -> None


## 3. Extract from real statement pages

In [4]:
import pymupdf, tempfile
from app.document_processing.parsers.pdf_parser import PdfParser
from app.financial_analytics.metric_extractor import extract_metrics

REPORT = "RIL-Integrated-Annual-Report-2024-25.pdf"     # any financial-statement PDF of yours
FIRST, LAST = 99, 100                                    # 1-based pages of the statements
src = pymupdf.open(nb.doc_path(REPORT))
sub = pymupdf.open(); sub.insert_pdf(src, from_page=FIRST - 1, to_page=LAST - 1)
tmp = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False).name; sub.save(tmp)
parsed = PdfParser().parse(tmp)
metrics = extract_metrics(parsed.elements, "pdf", doc_currency=None, doc_unit=None)
print(len(parsed.elements), "elements ->", len(metrics), "canonical metric values")

142 elements -> 68 canonical metric values


In [5]:
import pandas as pd
df = pd.DataFrame([{"metric": m.metric_key, "printed label": m.raw_label, "printed": m.raw_value, "value": str(m.value),
                    "unit": m.unit, "period": m.period_label, "statement": m.statement_type, "basis": m.basis,
                    "source": m.location.describe(), "confidence": m.confidence} for m in metrics])
df.head(15)

,metric,printed label,printed,value,unit,period,statement,basis,source,confidence
0,total_non_current_assets,Total Non-Current Assets,"14,50,851",1450851,crore,"As at 31st March, 2025",balance_sheet,consolidated,page 1 (printed p. 194),0.9
1,total_non_current_assets,Total Non-Current Assets,"12,85,886",1285886,crore,"As at 31st March, 2024",balance_sheet,consolidated,page 1 (printed p. 194),0.9
2,inventories,Inventories,"1,46,062",146062,crore,"As at 31st March, 2025",balance_sheet,consolidated,page 1 (printed p. 194),0.9
3,inventories,Inventories,"1,52,770",152770,crore,"As at 31st March, 2024",balance_sheet,consolidated,page 1 (printed p. 194),0.9
4,trade_receivables,Trade Receivables,"42,121",42121,crore,"As at 31st March, 2025",balance_sheet,consolidated,page 1 (printed p. 194),0.9
5,trade_receivables,Trade Receivables,"31,628",31628,crore,"As at 31st March, 2024",balance_sheet,consolidated,page 1 (printed p. 194),0.9
6,cash_and_cash_equivalents,Cash and Cash Equivalents,"1,06,502",106502,crore,"As at 31st March, 2025",balance_sheet,consolidated,page 1 (printed p. 194),0.9
7,cash_and_cash_equivalents,Cash and Cash Equivalents,"97,225",97225,crore,"As at 31st March, 2024",balance_sheet,consolidated,page 1 (printed p. 194),0.9
8,total_current_assets,Total Current Assets,"4,99,270",499270,crore,"As at 31st March, 2025",balance_sheet,consolidated,page 1 (printed p. 194),0.9
9,total_current_assets,Total Current Assets,"4,70,100",470100,crore,"As at 31st March, 2024",balance_sheet,consolidated,page 1 (printed p. 194),0.9


Each row is one value: printed form (`9,80,136`), exact `Decimal`, unit from the table's unit line, the column header as period,
statement and basis from the page title (*Consolidated Balance Sheet*), and the location of the **cell** itself.

> Page numbers here are relative to the two-page excerpt we parsed (page 1 = PDF page 99). The printed page labels (194–197)
> are the real ones. When the application processes the whole uploaded file, the PDF page numbers are the true ones too.

## 4. Structured JSON output
This is the shape stored in `financial_metrics` / `source_references` and returned by the API.

In [6]:
import json
m = next(m for m in metrics if m.metric_key == "total_assets")
record = {"metric_key": m.metric_key, "metric_name": m.metric_name, "raw_label": m.raw_label, "raw_value": m.raw_value,
          "value": str(m.value), "unit": m.unit, "unit_multiplier": m.unit_multiplier,
          "normalized_value": str(m.value * m.unit_multiplier) if m.unit_multiplier else None,
          "currency": m.currency, "period_label": m.period_label, "period_end": m.period_end.isoformat() if m.period_end else None,
          "statement_type": m.statement_type, "basis": m.basis, "location": m.location.to_dict(),
          "extraction_method": m.method, "confidence": m.confidence, "notes": m.notes}
print(json.dumps(record, indent=2, ensure_ascii=False))

{
  "metric_key": "total_assets",
  "metric_name": "Total assets",
  "raw_label": "Total Assets",
  "raw_value": "19,50,121",
  "value": "1950121",
  "unit": "crore",
  "unit_multiplier": 10000000,
  "normalized_value": "19501210000000",
  "currency": null,
  "period_label": "As at 31st March, 2025",
  "period_end": "2025-03-31",
  "statement_type": "balance_sheet",
  "basis": "consolidated",
  "location": {
    "file_format": "pdf",
    "page": 1,
    "page_label": "194",
    "half": "left",
    "bbox": [
      432.5,
      500.5,
      472.0,
      509.5
    ]
  },
  "extraction_method": "canonical-label match (pdf table row)",
  "confidence": 0.9,
  "notes": []
}


`currency` is `None`: this page states only "(C in crore)" — the ₹ glyph is font-encoded — so FinRAG does not assume a currency.
`normalized_value` multiplies by 10,000,000 (one crore) because the unit is known.

## 5. Traceability: compare with independently verified values
`docs/evaluation/gold_extraction_public_annual_report.jsonl` lists values a person read from the PDF. We compare the consolidated ones
with what we just extracted. (In the application, the same check runs as an `extraction_gold` evaluation.)

In [7]:
from decimal import Decimal
gold = [json.loads(l) for l in open(nb.PROJECT_DIR / "docs/evaluation/gold_extraction_public_annual_report.jsonl")]
rows = []
for g in gold:
    if g["basis"] != "consolidated":
        continue  # standalone values are on other pages
    cands = [m for m in metrics if m.metric_key == g["metric_key"] and g["period_contains"] in (m.period_label or "") and m.basis == "consolidated"]
    hit = next((m for m in cands if m.value == Decimal(g["expected_value"])), None)
    rows.append({"metric": g["metric_key"], "expected": g["expected_value"], "extracted": [str(m.value) for m in cands],
                 "correct": hit is not None, "source": hit.location.describe() if hit else None})
res = pd.DataFrame(rows)
print(f"{res.correct.sum()} of {len(res)} consolidated gold values extracted correctly from these two pages")
res

12 of 12 consolidated gold values extracted correctly from these two pages


,metric,expected,extracted,correct,source
0,total_assets,1950121,[1950121],True,page 1 (printed p. 194)
1,total_equity,1009626,[1009626],True,page 1 (printed p. 195)
2,total_liabilities,940495,[940495],True,page 1 (printed p. 195)
3,revenue_from_operations,980136,[980136],True,page 2 (printed p. 196)
4,net_profit,81309,[81309],True,page 2 (printed p. 196)
5,profit_before_tax,106017,[106017],True,page 2 (printed p. 196)
6,eps_basic,51.47,[51.47],True,page 2 (printed p. 197)
7,borrowings_current,110631,[110631],True,page 1 (printed p. 195)
8,borrowings_non_current,236899,[236899],True,page 1 (printed p. 195)
9,cash_and_cash_equivalents,106502,[106502],True,page 1 (printed p. 194)


## 6. When extraction is ambiguous: flag, don't guess
Some tables print two currencies or units under one year header (e.g. ₹ crore and US$ million side by side). FinRAG cannot know
which sub-column is which from the header alone, so it lowers confidence and adds a note. Page 40 of the saved report has such a table.

In [8]:
sub2 = pymupdf.open(); sub2.insert_pdf(src, from_page=39, to_page=39)
tmp2 = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False).name; sub2.save(tmp2)
amb = [m for m in extract_metrics(PdfParser().parse(tmp2).elements, "pdf", None, None) if m.metric_key == "profit_before_tax"]
for m in amb[:4]:
    print(f"{m.raw_value:>10} period={m.period_label!r:<10} confidence={m.confidence} notes={m.notes[-1][:90]}…")

    46,128 period='2024-25'  confidence=0.5 notes=several values in this row align with the same column header (sub-columns such as differen…
     5,397 period='2024-25'  confidence=0.5 notes=several values in this row align with the same column header (sub-columns such as differen…
    55,273 period='2023-24'  confidence=0.5 notes=several values in this row align with the same column header (sub-columns such as differen…
     6,627 period='2023-24'  confidence=0.5 notes=several values in this row align with the same column header (sub-columns such as differen…


## Interpretation
* All consolidated gold values on these pages were extracted with the exact printed figure and a cell-level location.
* Distinct definitions stay distinct; ambiguous labels are resolved only by explicit row-group context.
* Ambiguous layouts are *flagged* (low confidence + note), and validation later marks them `needs_review`.

## Common errors and troubleshooting
| Symptom | Cause | Fix |
|---|---|---|
| A metric is missing | Label wording not covered | Add a pattern to `METRICS` in `metric_extractor.py` + a test |
| Wrong period on a value | Header line not detected | See notebook 02 (`_header_columns`) |
| Unit `None` | No unit statement near the table and none for the document | Correct the document unit in Document Analysis |
| Same metric twice for one period | Several tables (highlights + statements) | Expected; validation and analytics prefer validated statement values |

## Practical exercises
1. Run the extraction on the statement pages of your own report. Which rows map to `None`? Should any of them map to a canonical metric?
2. Add a new canonical metric (e.g. *Dividend paid*) with a pattern and write a unit test for it.
3. Create a small gold file for your document and run the `extraction_gold` evaluation from the Evaluation page.

## Summary
You used the production extractor on real statements, saw how label matching and row groups work, produced the structured JSON
FinRAG stores, verified the result against hand-checked values, and saw how ambiguity is flagged instead of guessed.

## Further learning
* FinRAG source: `backend/app/financial_analytics/metric_extractor.py`
* Ind AS / IFRS presentation of financial statements (IAS 1): https://www.ifrs.org/issued-standards/list-of-standards/ias-1-presentation-of-financial-statements/
* Next: **07 — Financial validation**.